<span style="color:red">Work in progress</span>

--> This jupyter notebook serve as the basic structure to perform life cycle assessment within our research group (Used at least by Augustin & Robin).

# Introduction

The architecture of this notebook is explained in the Figure bellow that depicts the three different type of brightway2 (BW2) databases that are used in the notebook:
- the biosphere database from ecoinvent
- the original ecoinvent database version 3.9.1 cutoff
- an Open Source (OS) database where custom activities are created.

This jupyter notebook does the following:
1. Create the BW2 environment to compute the LCA
2. Generate the foreground activities, custom activities and modified activies in the framework of LCA algebraic using several excel file as data input
3. Compute LCIA results with uncertainty and distribution based on parameter distributions.

The following figure illustrates the database structure of this BW2 notebook and the purpose of the various files used.

![title](image/figure_notebook_structure.png)


# Importing relevant packages

In [1]:
import brightway2 as bw
import lca_algebraic as agb
from sympy import init_printing
import bw2io
from dotenv import load_dotenv
import pandas as pd
from sympy import symbols
import logging
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os 
import json
from importlib import reload
import yaml
from pathlib import Path


In [2]:
# Custom packages
from src.ei_access import EI_Access
from src.ei_access.setup import setup_ecoinvent_database

from src.utils.sheets_parser import find_activity,process_parameters,create_custom_activities,create_foreground

# Parameters

In [3]:

#### Project & ecoinvent database parameters
project_name= 'ECS-LCA'

ei_acc = EI_Access()

#### Life cycle data parameters
xlsx_path = "./sheets/"
yaml_path = "./yaml/"
# This Excel file must be formatted according to the provided template.
LC_data_Foreground_file_path = yaml_path + "foreground.yaml"
foreground_list=["foreground"] #These foreground can represent different scenario for example

#Specify the files that contain the data to build the custom and modified activities.
custom_meta_data_YAML = yaml_path + "custom_meta_data_YAML.xlsx"

# Specify the Foreground for which the results will be computed
selected_foreground = 'foreground' 

# Specify the LCIA method with which the results will be computed
impacts_GWP_specific=('EF v3.1', 'climate change', 'global warming potential (GWP100)')
#impacts_GWP_specific=('CML v4.8 2016 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')
impacts_ADP_specific=('EF v3.1', 'material resources: metals/minerals', 'abiotic depletion potential (ADP): elements (ultimate reserves)')
LCIA_method_list=[impacts_GWP_specific,impacts_ADP_specific]
#Specify amoung which label the contribution analysis should be performed:
contribution_analysis_label = "c_lc_phase"

OS_database="OS database"

#Specify the output folder to export the results
output_folder = r"./results"

In [4]:
bw.projects

Brightway2 projects manager with 3 objects:
	ECS-LCA
	TFE_ICT_draft_V1_independentTestNewEnv
	default
Use `projects.report()` to get a report on all projects.

# Initialize the project and the ecoinvent database

In [5]:
# Uncomment and run only if you think you messed up your project
#bw.projects.delete_project(name=project_name, delete_dir=True)

In [6]:
bw.projects.set_current(project_name) # Set the current project, can be any name

In [7]:
if 'biosphere3' not in bw.databases.keys():
    #Content of bw2.setup() function but with overwrite=True
    bw.create_default_biosphere3()
    bw.create_default_lcia_methods(overwrite=True)
    bw.create_core_migrations()

In [8]:
setup_ecoinvent_database(ei_acc)

Initial setup already done for ecoinvent-3.9.1-cutoff, skipping


In [9]:
agb.list_databases()

,backend,nb_activities,type
name,,,
biosphere3,sqlite,4709,biosphere
ecoinvent-3.9.1-biosphere,sqlite,4718,biosphere
ecoinvent-3.9.1-cutoff,sqlite,21238,background
OS database,sqlite,7,foreground


# Load the life-cycle data (foreground systems, created and modified activities)

In [10]:
# Loading the foreground systems in a dictionnary
foregrounds_All = {}
for foreground_name in foreground_list:
    with open(yaml_path + foreground_name + ".yaml", "r") as f:
        foreground = yaml.safe_load(f)
        foregrounds_All[foreground_name]=foreground
print(foregrounds_All)

{'foreground': [{'id': 'capa', 'c_lc_phase': 'prod', 'amount': {'parameter': 'num_capa', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}}, {'id': 'capa2', 'c_lc_phase': 'use', 'amount': {'parameter': 'num_capa2', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}}]}


In [11]:
custom_meta_data_YAML_DF=pd.read_excel(custom_meta_data_YAML)
custom_meta_data_YAML_DF=custom_meta_data_YAML_DF.set_index('file')
config = yaml.safe_load(open(yaml_path + "config.yaml"))

In [12]:
def die_area_pred(package_data):
    #supposes package in mm2, should translate to be robost
    if package_data["type"] == "BGA":
        return 0.822*package_data["area"]["value"]**0.73
    if package_data["type"] == "WLP":
        return 0.759*package_data["area"]["value"]**0.99
    raise Exception("Package type not supported")

def chip_smart_activity(activity):
    activity["output"] = {"product": activity["id"], "amount": 1, "unit":"unit"}

    die_area = activity["data"].get("die", {}).get("area", None)
    if die_area == None:
        package_data = activity["data"].get("package", None)
        if package_data == None:
            Exception(f"Not enough data to predict impact of chip {activity['id']}")
        die_area = {"value": die_area_pred(package_data), "unit": "mm2"}
    activity["inputs"] = [
        {
            "id": "wafer_die",
            "amount": {
                "parameter": "die_area",
                "typical": die_area["value"],
                "unit": die_area["unit"]
            },
            "maps_to": {
                "name": "market for wafer, fabricated, for integrated circuit"
            },
        },
    ]
    return activity
    
def smart_activity(activity):
    if activity["type"] == "chip":
        return chip_smart_activity(activity)
    raise Exception("Activity type not supported")

In [13]:
def load_custom_activities(yaml_path):
    custom_root = Path(yaml_path) / "custom"
    activities = []

    for file in custom_root.rglob("*.yaml"):
        with open(file, "r") as f:
            data = yaml.safe_load(f)
            print(data)
            for activity in data:
                # preserve category structure
                relative_path = file.relative_to(custom_root)
                activity["folder"] = relative_path.parent.as_posix()
                if "type" in activity:
                    activity = smart_activity(activity)
                activities.append(activity)

    return activities

In [14]:
def generate_parameter_registry(activities):

    with open(yaml_path + "config.yaml", "r") as f:
        config = yaml.safe_load(f)

    registry = {}

    uncertainty_cfg = config.get("uncertainty", {})
    global_default_value = uncertainty_cfg.get("default_value")
    global_default_distribution = uncertainty_cfg.get("default_distribution")

    for activity in activities:
        activity_id = activity["id"]

        for input_ in activity.get("inputs", []):

            amount = input_.get("amount")
            if not isinstance(amount, dict):
                continue

            param_name = amount["parameter"]
            namespaced = f"{activity_id}_{param_name}"

            if namespaced in registry:
                raise ValueError(f"Duplicate parameter: {namespaced}")

            if "typical" in amount:
                typical_value = amount["typical"]
                uncertainty_block = amount.get("uncertainty", {})
                if uncertainty_block:
                    distribution = uncertainty_block.get(
                        "distribution",
                        global_default_distribution
                    )
                else:
                    distribution = global_default_distribution
            else:
                typical_value = global_default_value
                distribution = global_default_distribution
            

            registry[namespaced] = {
                "source_activity": activity_id,
                "typical": typical_value,
                "distribution": distribution,
                "uncertainty": uncertainty_block,
            }

    return registry

In [15]:
def create_parameters(parameter_registry):
    ret = {}
    error_occurred = False

    for param_name, param_data in parameter_registry.items():
        print(param_data)
        typical = param_data.get("typical", 0)
        distribution = param_data.get("distribution", None)
        uncertainty = param_data.get("uncertainty", {})

        try:
            param = agb.newFloatParam(
                param_name,
                default=typical,
                min=uncertainty.get("min"),
                max=uncertainty.get("max"),
                std=uncertainty.get("std"),
                distrib=getattr(
                    agb.DistributionType,
                    distribution.upper(),
                    agb.DistributionType.FIXED
                ),
                description=uncertainty.get("description"),
                label=uncertainty.get("label"),
            )
            ret[param_name] = param
        except Exception as e:
            error_occurred = True
            print(f"Error creating parameter '{param_name}': {e}")
    
    if not error_occurred:
        print("All parameters have being created successfully.")
    return ret

In [16]:
custom_activities = load_custom_activities(yaml_path)
    
parameter_registry = generate_parameter_registry(custom_activities)
print(parameter_registry)
agb.resetParams()   
parameter_registry = create_parameters(parameter_registry)
print(parameter_registry)
sorted_sheets = (custom_meta_data_YAML_DF.sort_values(by="priority").index.tolist()) #increasing order

[{'id': 'resistor_smd_0805', 'name': 'Resistor SMD 0805', 'description': 'Surface mounted thick-film resistor 0805 format.\n', 'output': {'product': 'resistor_smd_0805', 'amount': 1, 'unit': 'unit'}, 'inputs': [{'id': 'copper_termination', 'amount': {'parameter': 'mass_copper', 'typical': 0.0003, 'unit': 'kg'}, 'maps_to': {'name': 'market for copper, cathode'}}]}]
[{'id': 'capacitor_smd_0603', 'name': 'Capacitor SMD 0603', 'description': 'Surface mounted ceramic capacitor 0603 format.\n', 'c_capacitor_value': '3nF', 'c_garbage': 83, 'output': {'product': 'capacitor_smd_0603', 'amount': 1, 'unit': 'unit'}, 'inputs': [{'id': 'ceramic_dielectric', 'amount': {'parameter': 'mass_ceramic', 'typical': 0.002, 'unit': 'kg'}, 'maps_to': {'name': 'ceramic tile production', 'location': 'RoW'}}, {'id': 'copper_electrode', 'amount': {'parameter': 'mass_copper', 'typical': 0.0005, 'unit': 'kg', 'uncertainty': {'distribution': 'normal', 'std': 5e-05}}, 'maps_to': {'name': 'market for copper, cathode',

In [17]:
# Micro sanity check ;-) It should print the parameter
agb.all_params()['capacitor_smd_0603_mass_ceramic']

capacitor_smd_0603_mass_ceramic

In [18]:


OS_database="OS database"

agb.resetDb(OS_database)
agb.setForeground(OS_database) #Create one database where all custom and modified activities will be added.
agb.list_databases()        # Sanity check - All activities reset to zero ?



[WARNING] Db OS database was here. Reseting it


,backend,nb_activities,type
name,,,
biosphere3,sqlite,4709,biosphere
ecoinvent-3.9.1-biosphere,sqlite,4718,biosphere
ecoinvent-3.9.1-cutoff,sqlite,21238,background
OS database,sqlite,0,foreground


In [19]:
def create_custom_activities(custom_activities, parameter_registry, config, foreground_db):

    default_location = config["background"]["default_location"]
    print(foreground_db)
    for activity in custom_activities:
        print(activity)
        activity_id = activity["id"]
        unit = activity["output"]["unit"]
        print(f"Creating activity: {activity_id}")
        exchanges = {}

        for input_ in activity.get("inputs", []):

            amount_block = input_.get("amount")
            if not isinstance(amount_block, dict):
                continue

            param_name = amount_block["parameter"]
            namespaced_param = f"{activity_id}_{param_name}"

            if namespaced_param not in parameter_registry:
                raise ValueError(
                    f"Parameter {namespaced_param} not found in registry."
                )
            #print(namespaced_param)
            # Retrieve created parameter object
            parameter_obj = parameter_registry[namespaced_param]

            # Resolve mapping
            maps_to = input_["maps_to"]
            ei_name = maps_to["name"]
            location = maps_to.get("location", default_location)

            # Find background activity
            ei_activity = agb.findTechAct(ei_name,loc=location,)

            if ei_activity is None:
                raise ValueError(
                    f"Background activity not found: {ei_name} ({location})"
                )
            #print(parameter_obj)
            exchanges[ei_activity] =  agb.all_params()[namespaced_param]  #eval(namespaced_param) # actual project parameter !!!

        # Create new custom activity
        try:
            #print(exchanges)
            agb.newActivity(
                foreground_db,
                activity_id,
                unit,
                exchanges=exchanges
            )
            print(f"Activity created: {activity_id}")

        except Exception as e:
            print(f"Error creating activity '{activity_id}': {e}")

    print("All custom activities processed.")

# Create custom activities

In [20]:
create_custom_activities(custom_activities, parameter_registry, config, foreground_db=OS_database)
print(parameter_registry)

OS database
{'id': 'resistor_smd_0805', 'name': 'Resistor SMD 0805', 'description': 'Surface mounted thick-film resistor 0805 format.\n', 'output': {'product': 'resistor_smd_0805', 'amount': 1, 'unit': 'unit'}, 'inputs': [{'id': 'copper_termination', 'amount': {'parameter': 'mass_copper', 'typical': 0.0003, 'unit': 'kg'}, 'maps_to': {'name': 'market for copper, cathode'}}], 'folder': 'passive'}
Creating activity: resistor_smd_0805
Activity created: resistor_smd_0805
{'id': 'capacitor_smd_0603', 'name': 'Capacitor SMD 0603', 'description': 'Surface mounted ceramic capacitor 0603 format.\n', 'c_capacitor_value': '3nF', 'c_garbage': 83, 'output': {'product': 'capacitor_smd_0603', 'amount': 1, 'unit': 'unit'}, 'inputs': [{'id': 'ceramic_dielectric', 'amount': {'parameter': 'mass_ceramic', 'typical': 0.002, 'unit': 'kg'}, 'maps_to': {'name': 'ceramic tile production', 'location': 'RoW'}}, {'id': 'copper_electrode', 'amount': {'parameter': 'mass_copper', 'typical': 0.0005, 'unit': 'kg', 'unc

In [21]:
# Small sanity check - to be removed later
agb.list_databases() #Should be 2 for now in the OS database


,backend,nb_activities,type
name,,,
biosphere3,sqlite,4709,biosphere
ecoinvent-3.9.1-biosphere,sqlite,4718,biosphere
ecoinvent-3.9.1-cutoff,sqlite,21238,background
OS database,sqlite,3,foreground


In [22]:
agb.printAct(agb.findActivity('capacitor_smd_0603', db_name= OS_database))

capacitor_smd_0603 (1.000000 unit)  \
                                                                                 input   
ceramic tile production                                   ceramic tile production[RoW]   
market for copper, cathode                                  market for copper, cathode   
market for tantalum concentrate, 30% Ta2O5  market for tantalum concentrate, 30% Ta2O5   

                                                                              \
                                                                      amount   
ceramic tile production                      capacitor_smd_0603_mass_ceramic   
market for copper, cathode                    capacitor_smd_0603_mass_copper   
market for tantalum concentrate, 30% Ta2O5  capacitor_smd_0603_mass_tantalum   

                                                      
                                                unit  
ceramic tile production                     kilogram  
market for copper, cathode                  kilogram  
market for tantalum concentrate, 30% Ta2O5  kilogram

In [23]:
# Small sanity check - to be removed later
absolute_impacts = agb.compute_impacts(agb.findActivity('capacitor_smd_0603', db_name= OS_database), LCIA_method_list, functional_unit= 1)
absolute_impacts

[INFO] Db changed recently, clearing cache lcia
[INFO] Required param 'capacitor_smd_0603_mass_copper' was missing, replacing by default value : 0.0005
[INFO] Required param 'capacitor_smd_0603_mass_copper' was missing, replacing by default value : 0.0005


,climate change - global warming potential (GWP100)[kg CO2-Eq],material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)[kg Sb-Eq]
capacitor_smd_0603,0.00838303,3.76725e-06


# Defining the foreground

In [24]:
#agb.resetDb(OS_database)
agb.setForeground(OS_database) #Create one database where all custom and modified activities will be added.
agb.list_databases()        # Sanity check - All activities reset to zero ?

,backend,nb_activities,type
name,,,
biosphere3,sqlite,4709,biosphere
ecoinvent-3.9.1-biosphere,sqlite,4718,biosphere
ecoinvent-3.9.1-cutoff,sqlite,21238,background
OS database,sqlite,3,foreground


In [25]:
def get_param_type(value):
    if isinstance(value, bool):
        return "boolean"
    elif isinstance(value, float):
        return "float"
    elif isinstance(value, int):
        return "float"
    elif isinstance(value, str):
        return "enum"
    else:
        raise ValueError(f"Unsupported type: {typenum_capa(value)}")
def process_parameters(params_df, parameter_registry,sheet_name):
    """
    Create the parameters in the lca algebraic framework for one of the excel sheet.

    params_df: the dataframe that has been created for one excel sheet
    parameter_registry: the register of parameter
    sheet_name: the name of the excel sheet params_df
    """
    print(params_df)

    for index, row in enumerate(params_df):
        param_name = f"{sheet_name}_{index}"
        param_type = get_param_type(row["amount"]["typical"]).strip().lower()

        try:
            if param_type == "float":
                print(f"{row} is a float!")
                param = agb.newFloatParam(
                    row["amount"]["parameter"],
                    default=row["amount"]["typical"],
                    min=row["amount"].get("uncertainty",{}).get("min"),
                    max=row["amount"].get("uncertainty",{}).get("max"),
                    std=row["amount"].get("uncertainty",{}).get("std"),
                    distrib=getattr(agb.DistributionType, row["amount"].get("uncertainty",{}).get("distribution", "FIXED").upper(), None),
                    description=row.get("Description"),
                    label=row.get("Label")
                )
            elif param_type == "bool":
                param = agb.newBoolParam(
                    param_name,
                    default=row["output"]["value"]
                )
            elif param_type == "enum":
                values = eval(row["Values"]) if isinstance(row["Values"], str) else row["Values"]
                weights = eval(row["Weights"]) if isinstance(row["Weights"], str) else None

                if weights:
                    values = {k: v for k, v in zip(values, weights)}

                param = agb.newEnumParam(
                    param_name,
                    values=values,
                    default=row["Default"],
                    description=row.get("Description")
                )
            else:
                raise ValueError(f"Unsupported parameter type: {param_type}")

            # Register the parameter for later evaluation
            print(param_type, param)
            parameter_registry[param_name] = param
            print(f"Parameter created and registered: {param_name}") # for debugging only !
        except Exception as e:
            print(f"Error creating parameter '{param_name}': {e}")


In [26]:
# still the function for excel, 
process_parameters(foregrounds_All[f"{selected_foreground}"], parameter_registry,selected_foreground) # create new parameters for foreground activities and add them to the parameter_registry
print(parameter_registry)

[{'id': 'capa', 'c_lc_phase': 'prod', 'amount': {'parameter': 'num_capa', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}}, {'id': 'capa2', 'c_lc_phase': 'use', 'amount': {'parameter': 'num_capa2', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}}]
{'id': 'capa', 'c_lc_phase': 'prod', 'amount': {'parameter': 'num_capa', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}} is a float!
float num_capa
Parameter created and registered: foreground_0
{'id': 'capa2', 'c_lc_phase': 'use', 'amount': {'parameter': 'num_capa2', 'typical': 1, 'unit': 'kg'}, 'maps_to': {'name': 'capacitor_smd_0603'}} is a float!
float num_capa2
Parameter created and registered: foreground_1
{'resistor_smd_0805_mass_copper': resistor_smd_0805_mass_copper, 'capacitor_smd_0603_mass_ceramic': capacitor_smd_0603_mass_ceramic, 'capacitor_smd_0603_mass_copper': capacitor_smd_0603_mass_copper, 'capacitor_smd_0603_mass_tantalum': capacitor_smd_0603_mass_tantalu

In [27]:
def create_foreground(foregrounds,selected_foreground,OS_database,parameter_registry):
    
    default_location = config["background"]["default_location"]
    for index, row in enumerate(foregrounds):          # Iterate through the rows of the foreground sheet and create new activities

        ei_activity_name =   "UCLouvain custom process" #Todo rework find_activity     # Read values from the Excel file
        location = row["maps_to"].get("loc")                            # Location (optional)
        new_activity_name = f"priv_lca_{row['id']}"    # the name of the new activity is defined in "LCA algebraic name"
        parameter_expression = f"{selected_foreground}_{index}"          # Name of the associated parameter
        modified_activity_name = None#row["Modified process name"]         # name of the modified activity (if relevant)
        custom_process_name = row["maps_to"]["name"] # name of the custom activity (if relevant)


        maps_to = row["maps_to"]
        ei_name = maps_to["name"]
        location = maps_to.get("location", default_location)

        # Find background activity
        ei_activity = agb.findActivity(custom_process_name,loc=location,db_name=OS_database)
        
        if ei_activity is None:
            print(f"Skipping creation of {new_activity_name} due to unresolved activity issues.")
            continue

        try:
            exchanges = {ei_activity: eval(parameter_expression, {}, parameter_registry)}  # Define exchanges
            print(exchanges, parameter_expression, parameter_registry)
        except Exception as e:
            print(f"Error in parameter expression for activity '{new_activity_name}': {e}")
            continue

        try:
            agb.newActivity(OS_database, new_activity_name, "unit", exchanges=exchanges)
            print(f"Activity created: {new_activity_name}")
        except Exception as e:
            print(f"Error creating activity '{new_activity_name}': {e}")


In [28]:
create_foreground(foregrounds_All[f"{selected_foreground}"],selected_foreground,OS_database,parameter_registry)
print(parameter_registry)

{'capacitor_smd_0603' (unit, GLO, None): num_capa} foreground_0 {'resistor_smd_0805_mass_copper': resistor_smd_0805_mass_copper, 'capacitor_smd_0603_mass_ceramic': capacitor_smd_0603_mass_ceramic, 'capacitor_smd_0603_mass_copper': capacitor_smd_0603_mass_copper, 'capacitor_smd_0603_mass_tantalum': capacitor_smd_0603_mass_tantalum, 'my_chip_die_area': my_chip_die_area, 'foreground_0': num_capa, 'foreground_1': num_capa2}
Activity created: priv_lca_capa
{'capacitor_smd_0603' (unit, GLO, None): num_capa2} foreground_1 {'resistor_smd_0805_mass_copper': resistor_smd_0805_mass_copper, 'capacitor_smd_0603_mass_ceramic': capacitor_smd_0603_mass_ceramic, 'capacitor_smd_0603_mass_copper': capacitor_smd_0603_mass_copper, 'capacitor_smd_0603_mass_tantalum': capacitor_smd_0603_mass_tantalum, 'my_chip_die_area': my_chip_die_area, 'foreground_0': num_capa, 'foreground_1': num_capa2}
Activity created: priv_lca_capa2
{'resistor_smd_0805_mass_copper': resistor_smd_0805_mass_copper, 'capacitor_smd_0603_m

# Foreground generation

In [29]:
Exchanges_Foreground = {} # Initialize an empty dictionary to store exchanges

for index, row in enumerate(foregrounds_All[f"{selected_foreground}"]): # Iterate thmodifiérough the main sheet to collect activities for the exchanges
    Exchanges_Foreground[agb.findActivity(f"priv_lca_{row['id']}" , db_name=OS_database, loc="GLO")]=1

print(Exchanges_Foreground)
agb.newActivity(OS_database,selected_foreground,  "unit", exchanges=Exchanges_Foreground) # Create the foreground

{'priv_lca_capa' (unit, GLO, None): 1, 'priv_lca_capa2' (unit, GLO, None): 1}


'foreground' (unit, GLO, None)

In [30]:
total_foreground_exchanges = {}
total_foreground_exchanges[agb.findActivity(selected_foreground, db_name=OS_database, loc="GLO")] = 1

# Reference Flow

In [31]:
reference_flow = agb.newActivity(OS_database, "reference flow", "unit", exchanges=total_foreground_exchanges)

# Impact Methods Choice

In [32]:
# Retrieve impact categories (here from EF v3.1)
impacts_GWP = agb.findMethods(search="climate change", mainCat="EF v3.1") # GWP impact categories from EF 3.1
impacts_GWP_other = agb.findMethods(search="climate change") # GWP impact categories from EF 3.1
impacts_ADP = agb.findMethods(search="ADP", mainCat="EF v3.1") # GWP impact categories from EF 3.1
impacts_all = agb.findMethods(search="", mainCat="EF v3.1")               # All impact categories from EF 3.1

# Compute Absolute Impacts

In [33]:
absolute_impacts = agb.compute_impacts(reference_flow, LCIA_method_list, functional_unit= 1)
absolute_impacts

[INFO] Db changed recently, clearing cache expr
[INFO] Db changed recently, clearing cache lcia
[INFO] Required param 'capacitor_smd_0603_mass_copper' was missing, replacing by default value : 0.0005
[INFO] Required param 'capacitor_smd_0603_mass_copper' was missing, replacing by default value : 0.0005


,climate change - global warming potential (GWP100)[kg CO2-Eq],material resources: metals/minerals - abiotic depletion potential (ADP): elements (ultimate reserves)[kg Sb-Eq]
reference flow,0.0167661,7.53451e-06


# Contribution Analysis

In [34]:
for row in foregrounds_All[f"{selected_foreground}"]:
    activity_name = f"priv_lca_{row['id']}"
    sub_assembly_label = row[contribution_analysis_label]  # choosing labels for activities -> life cycle phases : results per phase, 
    print(sub_assembly_label)
    act = agb.findActivity(activity_name, db_name=OS_database, loc="GLO")
    print(act)
    act.updateMeta(phase=sub_assembly_label, label=sub_assembly_label)
print("Finished labeling activities.")

print(reference_flow)

df_impacts_axis = agb.compute_impacts(reference_flow, LCIA_method_list, functional_unit=1, axis="label")
df_impacts_axis_ = df_impacts_axis.drop(index=['*sum*'])     # Drop the 'sum' 
df_sums = df_impacts_axis_.sum(axis=0)                       # Compute sums 
df_normalized = df_impacts_axis_.div(df_sums, axis=1) * 100  # Normalize data

[INFO] Db changed recently, clearing cache expr


prod
'priv_lca_capa' (unit, GLO, None)
use
'priv_lca_capa2' (unit, GLO, None)
Finished labeling activities.
'reference flow' (unit, GLO, None)


SympifyError: SympifyError: "cannot sympify object of type <class 'function'>"

In [ ]:
### ------------------------------------------------------------------
# 1.  Set-up
# ------------------------------------------------------------------
df = df_impacts_axis.drop(index="*sum*")                    # convenience alias
labels = df.index                       # contribution labels (rows)
categories = df.columns                 # the two impact categories
palette = sns.color_palette("Spectral", n_colors=len(labels))

# ------------------------------------------------------------------
# 2.  Create a 1-by-N grid of sub-plots (one per impact category)
# ------------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(categories),
    figsize=(10, 3),
    sharey=True,                 # keep the single y tick aligned
    constrained_layout=True
)

# ------------------------------------------------------------------
# 3.  Draw a stacked horizontal bar in *each* subplot
# ------------------------------------------------------------------
for ax, cat in zip(axes, categories):
    left = 0
    for color, (label, value) in zip(palette, df[cat].items()):
        ax.barh(
            y=0,                 # only one device
            width=value,
            left=left,
            height=0.6,
            color=color,
            label=label
        )
        left += value            # accumulate for stacking

    # --- Cosmetics -------------------------------------------------
    # Shorten the subplot title to everything before the first " - "
    ax.set_title(cat.split(' - ')[0], fontsize=11)

    # Label the x-axis with the unit text (text inside the brackets)
    unit = cat.split('[')[-1].rstrip(']')
    ax.set_xlabel(unit)
    ax.set_yticks([0])

# ------------------------------------------------------------------
# 4.  One shared legend and overall title
# ------------------------------------------------------------------
fig.legend(labels, loc='upper right', ncol=1, bbox_to_anchor=(0.97, 0.9))
#fig.suptitle('Contribution analysis — single device', y=1.15, fontsize=14)
#plt.savefig(os.path.join(output_folder,"fig_"+contribution_analysis_label+"_"+selected_foreground+".png"))
plt.show()

In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(12, 10))
dark_palette = sns.color_palette("Spectral", n_colors=len(df_normalized.index))
df_normalized.T.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=dark_palette)
#dark_palette = sns.color_palette("Spectral", n_colors=len(df_impacts_axis_.index))
#df_impacts_axis_.T.plot(kind='bar', stacked=True, ax=ax, width=0.8, color=dark_palette)

# Customize plot
ax.set_title('Impact Categories with Contributions from Selected Labels (Normalized to 100%)')
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(df_impacts_axis_.columns, rotation=45, ha='right')
ax.legend(title="Labels", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
#plt.savefig(os.path.join(output_folder,"figure","fig_normalized_"+contribution_analysis_label+"_"+selected_foreground+".png"))
plt.show()

# Monte-Carlo Analysis

In [ ]:
MC_runs=agb.incer_stochastic_violin(reference_flow, LCIA_method_list, functional_unit=1, figspace=(0.5,0.5), n=512, figsize=(8, 8), sharex=True,  nb_cols=2)

# Exporting results

In [ ]:
#output_file = os.path.join(output_folder,  selected_foreground+"_contribution_results.xlsx")
#df_impacts_axis.to_excel(output_file, index=True)
#print(f"Impact saved to {output_file}")